<img src="images/form_factors.png" alt="The AI Maturity Ladder — Five Form Factors of AI Applications" width="900">


# Evaluating the AI Maturity Ladder — with LangSmith

A companion to the **AI Maturity Ladder** notebook, where we built five form factors (chatbot → RAG → workflow → agent → autonomous agent). This notebook answers the question that decides whether any of them is production-ready: **is it actually any good — and how would you know if a prompt or model change made it worse?**

We use **LangSmith** as the evaluation platform and **Claude** as the judge.

> 💡 **Key Insight — you can't improve what you can't measure.** Every rung of the ladder fails in its own way: RAG retrieves the wrong document, a workflow misclassifies, an agent picks the wrong tool. Evaluation turns vague "this feels off" impressions into **numbers you can track, compare, and regression-test** every time you touch a prompt, a model, or a retrieval setting.

## What You Will Learn

- The vocabulary of LLM evaluation: **datasets, targets, evaluators, experiments, LLM-as-judge**
- Evaluate **retrieval** with classic information-retrieval metrics (recall@k, hit rate, MRR)
- Evaluate **RAG generation** for **correctness** and **groundedness** using LangSmith + a Claude judge
- Evaluate a **workflow classifier** with exact-match scoring
- Run a **rigorous retrieval bake-off** on the **BEIR** benchmark, scored with the classic IR metrics — **precision@k, recall@k, NDCG@k**
- Read an experiment in the LangSmith UI and use it for **regression testing**

## The Evaluation Landscape

> 💡 **Key Term — LangSmith:** A platform for tracing and evaluating LLM applications. Its core objects:
> - **Dataset** — a collection of *examples*. Each example has `inputs` (what you feed the app) and, usually, reference `outputs` (the "golden" answer to grade against).
> - **Target** — the function under test. LangSmith runs it on every example.
> - **Evaluator** — a function that scores the target's output. It can be a simple heuristic (exact match) or an **LLM-as-judge**.
> - **Experiment** — one run of *target + evaluators* over a dataset, with per-example scores you can browse, aggregate, and compare across versions.

> 💡 **Key Term — LLM-as-judge:** Using an LLM to *score* another model's output against a rubric ("is this answer faithful to the provided context?"). It scales the subjective quality checks that exact-match can't capture. Here the judge is **Claude**, called through the `anthropic` SDK and constrained to return a structured `{score, reason}` verdict.

## Setup

> **Note:** Installs everything this notebook needs. Skip it if your environment is already prepared. We reuse the same Oracle + `fastembed` stack as the companion notebook, and add **`langsmith`** for evaluation.

In [1]:
# - anthropic   -> the model under test, and the LLM-as-judge
# - langsmith   -> datasets, experiments, evaluators, tracing
# - oracledb + fastembed + pandas + numpy -> the Oracle-backed RAG system under test

# %pip install -Uq anthropic langsmith oracledb fastembed pandas numpy

## Configure Access

Two credentials: `ANTHROPIC_API_KEY` (the model + judge) and `LANGSMITH_API_KEY` (the platform — create a free key at [smith.langchain.com](https://smith.langchain.com)). Setting `LANGSMITH_TRACING=true` makes every Claude call show up as a trace, and `wrap_anthropic` is what wires the SDK into LangSmith.

In [2]:
import os
import json
import anthropic
from langsmith import Client
from langsmith.wrappers import wrap_anthropic

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY is not set."
assert os.environ.get("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY is not set — get one at https://smith.langchain.com."
os.environ.setdefault("LANGSMITH_TRACING", "true")        # send traces + experiments to LangSmith
os.environ.setdefault("LANGSMITH_PROJECT", "ai-maturity-evaluation")

MODEL = "claude-opus-4-8"
MAX_TOKENS = 1024

# wrap_anthropic auto-traces every Claude call into LangSmith (no-op if tracing is off).
client = wrap_anthropic(anthropic.Anthropic())
ls = Client()

# Confirm the LangSmith key is valid by making one lightweight authenticated call.
list(ls.list_datasets(limit=1))
print("Anthropic + LangSmith clients ready ✓")


def text_of(response) -> str:
    """Concatenate the text blocks of a Claude response into a single string."""
    return "".join(b.text for b in response.content if b.type == "text")

KeyboardInterrupt: 

# The System Under Test

Evaluation needs something to evaluate. We rebuild — compactly — the Oracle-backed **RAG pipeline** and the **workflow classifier** from the AI Maturity Ladder. This is plumbing, not the lesson; the full teaching walkthrough lives in the companion notebook. Here we just need a system to *measure*.

**Connect to Oracle and define the knowledge base** (the same dozen Acme Cloud docs).

In [ ]:
import oracledb
import time
import numpy as np
import array

DOCS = [
    {"doc_id": "plans", "title": "Plans & Pricing", "category": "billing",
     "content": "Acme Cloud offers three plans. Free includes 1 project and community support. "
                "Pro is $49 per user per month with email support. Enterprise has custom pricing, "
                "SSO, and a dedicated support engineer."},
    {"doc_id": "rate_limits", "title": "API Rate Limits", "category": "api",
     "content": "Acme Cloud enforces API rate limits per plan. Free allows 60 requests per minute; "
                "Pro allows 1,000 requests per minute; Enterprise limits are negotiated per contract."},
    {"doc_id": "upgrade", "title": "Upgrading Your Plan", "category": "billing",
     "content": "To upgrade, open Settings, then Billing, then Change Plan. Upgrades take effect "
                "immediately and are pro-rated for the current billing cycle."},
    {"doc_id": "regions", "title": "Data Residency & Regions", "category": "data",
     "content": "Acme Cloud stores data in US, EU (Frankfurt), and APAC (Singapore) regions. "
                "The region is chosen at project creation and cannot be changed afterward."},
    {"doc_id": "sla", "title": "Service Level Agreement", "category": "reliability",
     "content": "Acme Cloud guarantees 99.9% uptime on the Pro plan and 99.99% on Enterprise. "
                "SLA credits are issued automatically when monthly uptime falls below target."},
    {"doc_id": "support", "title": "Support Channels & Response Times", "category": "support",
     "content": "Free plan customers use the community forum. Pro email support responds within one "
                "business day. Enterprise includes 24/7 support with a one-hour response target."},
    {"doc_id": "api_keys", "title": "Creating & Rotating API Keys", "category": "api",
     "content": "Create and rotate API keys under Settings, then API Keys. Rotating a key immediately "
                "revokes the previous one, so update your applications before rotating."},
    {"doc_id": "sso", "title": "Single Sign-On (SSO)", "category": "security",
     "content": "SSO is available on the Enterprise plan and supports SAML 2.0 and OIDC. An "
                "administrator configures the identity provider under Settings, then Security, then SSO."},
    {"doc_id": "webhooks", "title": "Webhooks", "category": "integrations",
     "content": "Acme Cloud can send webhooks on project events. Configure endpoint URLs under "
                "Settings, then Webhooks. Failed deliveries are retried with exponential backoff."},
    {"doc_id": "backups", "title": "Backups & Recovery", "category": "data",
     "content": "Acme Cloud takes automated daily backups retained for 30 days on Pro and 90 days on "
                "Enterprise. Point-in-time recovery is available on the Enterprise plan."},
    {"doc_id": "roles", "title": "Team Roles & Permissions", "category": "account",
     "content": "Acme Cloud supports Owner, Admin, Member, and Viewer roles. Only Owners and Admins "
                "can manage billing, invite teammates, or rotate API keys."},
    {"doc_id": "export", "title": "Exporting Your Data", "category": "data",
     "content": "You can export project data as JSON or CSV from Settings, then Export. Large exports "
                "are emailed as a downloadable archive when ready."},
]


def connect_to_oracle(max_retries=3, retry_delay=5):
    for attempt in range(1, max_retries + 1):
        try:
            conn = oracledb.connect(
                user=os.environ.get("ORACLE_USER", "VECTOR"),
                password=os.environ.get("ORACLE_PASSWORD", "VectorPwd_2025"),
                dsn=os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1"))
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%'")
                print("Connected:", cur.fetchone()[0])
            return conn
        except oracledb.OperationalError as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(retry_delay)
            else:
                raise


conn = connect_to_oracle()
print(len(DOCS), "documents")

**Embed, create the table + indexes, and ingest** (`fastembed` / nomic, stored as native `VECTOR`).

In [ ]:
from fastembed import TextEmbedding

embedder = TextEmbedding(model_name="nomic-ai/nomic-embed-text-v1.5")


def _unit(v):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v / n if n else v


doc_vectors = np.array([_unit(v) for v in embedder.embed([f"{d['title']}. {d['content']}" for d in DOCS])],
                       dtype=np.float32)
dim = int(doc_vectors.shape[1])

table_ddl = f"""
BEGIN EXECUTE IMMEDIATE 'DROP TABLE acme_docs CASCADE CONSTRAINTS PURGE';
EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
CREATE TABLE acme_docs (doc_id VARCHAR2(64) PRIMARY KEY, title VARCHAR2(400),
    category VARCHAR2(64), content VARCHAR2(4000), embedding VECTOR({dim}, FLOAT32))
"""
with conn.cursor() as cur:
    for stmt in table_ddl.split("/"):
        if stmt.strip():
            cur.execute(stmt)
    try:
        cur.execute("DROP INDEX acme_text_idx")
    except oracledb.DatabaseError:
        pass
    cur.execute("CREATE INDEX acme_text_idx ON acme_docs(content) "
                "INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')")
    cur.executemany(
        "INSERT INTO acme_docs (doc_id, title, category, content, embedding) VALUES (:1,:2,:3,:4,:5)",
        [(d["doc_id"], d["title"], d["category"], d["content"], array.array("f", v))
         for d, v in zip(DOCS, doc_vectors.astype(np.float32).tolist())],
    )
conn.commit()
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM acme_docs")
    print(f"Ingested {cur.fetchone()[0]} rows at VECTOR dim={dim}.")

**The RAG pipeline and the workflow classifier** — the two systems we'll grade. `rag_pipeline` returns both the answer *and* the context it used, so we can score **groundedness** later.

In [ ]:
def embed_query(text: str):
    return array.array("f", _unit(next(embedder.query_embed(text))).astype(np.float32).tolist())


def retrieve(question: str, k: int = 3):
    """Vector search → [(content, similarity), ...]."""
    sql = f"""SELECT content, ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS sim
              FROM acme_docs ORDER BY sim DESC FETCH FIRST {int(k)} ROWS ONLY"""
    with conn.cursor() as cur:
        cur.execute(sql, q=embed_query(question))
        return [(c, float(s)) for c, s in cur.fetchall()]


def rag_pipeline(question: str, k: int = 3):
    """The RAG system under test: retrieve, then generate. Returns (answer, context)."""
    hits = retrieve(question, k)
    context = "\n".join(f"[{i + 1}] {doc}" for i, (doc, _) in enumerate(hits))
    system = (
        "You are the Acme Cloud support assistant. Answer using ONLY the context below; cite with "
        "[n]. If the answer isn't in the context, say you don't have that information.\n\n"
        f"Context:\n{context}"
    )
    resp = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, system=system,
                                  messages=[{"role": "user", "content": question}])
    return text_of(resp), context


CLASSIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": ["billing", "technical", "account", "feature_request", "other"]},
        "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": ["category", "urgency"],
    "additionalProperties": False,
}


def classify(message: str) -> dict:
    """The workflow classifier under test → {category, urgency}."""
    resp = client.messages.create(
        model=MODEL, max_tokens=256,
        system="Classify the incoming Acme Cloud support message.",
        messages=[{"role": "user", "content": message}],
        output_config={"format": {"type": "json_schema", "schema": CLASSIFY_SCHEMA}},
    )
    return json.loads(text_of(resp))


answer, ctx = rag_pipeline("What is the API rate limit on the Pro plan?")
print("RAG answer:", answer)
print("Classifier:", classify("I was double charged and need this fixed today!"))

# Part 1 — Evaluating Retrieval

> 💡 **Key Term — retrieval metrics:** Before judging the *answer*, judge the *evidence*. If the right document never makes the shortlist, no amount of LLM skill can save the answer. The standard metrics:
> - **Hit rate / Recall@k** — did a relevant doc land in the top-k?
> - **MRR (Mean Reciprocal Rank)** — how *high* did the first relevant doc rank (`1 / rank`)? Rewards putting the right doc first.

We build a small **labeled set**: each question is paired with the `doc_id`(s) that *should* be retrieved. The questions are paraphrased so that lexical overlap is low — which is exactly where keyword and vector search diverge.

In [ ]:
RETRIEVAL_EVALSET = [
    {"question": "How many API calls per minute do I get on the Pro tier?", "relevant": ["rate_limits"]},
    {"question": "How do I turn on single sign-on for my company?",          "relevant": ["sso"]},
    {"question": "What uptime do you promise?",                              "relevant": ["sla"]},
    {"question": "How do I move from the free tier to a paid one?",          "relevant": ["upgrade", "plans"]},
    {"question": "Which country is my information physically kept in?",      "relevant": ["regions"]},
    {"question": "How far back can I restore my project?",                   "relevant": ["backups"]},
    {"question": "A key leaked — how do I cycle it safely?",                 "relevant": ["api_keys"]},
    {"question": "Who is allowed to change the credit card on file?",        "relevant": ["roles"]},
    {"question": "Can I download everything as a spreadsheet?",              "relevant": ["export"]},
    {"question": "How quickly will someone reply if I email support?",       "relevant": ["support"]},
]
print(len(RETRIEVAL_EVALSET), "labeled retrieval examples")

**Three retrievers that return ranked `doc_id`s.** Keyword search over free-text questions needs the query reduced to its content words (Oracle Text's `CONTAINS` is a boolean/phrase language, not a sentence parser) — `_terms` does that. Vector search takes the raw question; hybrid fuses the two with Reciprocal Rank Fusion.

In [ ]:
import re

_STOP = {"the", "a", "an", "how", "do", "i", "my", "is", "are", "can", "get", "of", "on", "to",
         "in", "for", "what", "where", "does", "with", "me", "some", "you", "your", "from", "and", "if"}


def _terms(q: str) -> str:
    """Reduce a free-text question to OR-joined content words, safe for Oracle Text CONTAINS."""
    toks = [t for t in re.findall(r"[A-Za-z0-9]+", q.lower()) if len(t) > 2 and t not in _STOP]
    return " OR ".join("{" + t + "}" for t in toks) or "{none}"


def retr_keyword(q, k=5):
    sql = f"""SELECT doc_id FROM acme_docs WHERE CONTAINS(content, :kw, 1) > 0
              ORDER BY SCORE(1) DESC FETCH FIRST {int(k)} ROWS ONLY"""
    with conn.cursor() as cur:
        cur.execute(sql, kw=_terms(q))
        return [r[0] for r in cur.fetchall()]


def retr_vector(q, k=5):
    sql = f"""SELECT doc_id FROM acme_docs
              ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE) FETCH FIRST {int(k)} ROWS ONLY"""
    with conn.cursor() as cur:
        cur.execute(sql, q=embed_query(q))
        return [r[0] for r in cur.fetchall()]


def retr_hybrid(q, k=5, per_list=10, rrf_k=60):
    sql = f"""
        WITH vec AS (SELECT doc_id, ROW_NUMBER() OVER (ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE)) r
                     FROM acme_docs ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE) FETCH FIRST {int(per_list)} ROWS ONLY),
             txt AS (SELECT doc_id, ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) r
                     FROM acme_docs WHERE CONTAINS(content, :kw, 1) > 0
                     ORDER BY SCORE(1) DESC FETCH FIRST {int(per_list)} ROWS ONLY),
             fused AS (SELECT COALESCE(v.doc_id, t.doc_id) doc_id, NVL(v.r, 999999) rv, NVL(t.r, 999999) rt
                       FROM vec v FULL OUTER JOIN txt t ON t.doc_id = v.doc_id)
        SELECT doc_id FROM fused ORDER BY (1.0/(:rk + rv) + 1.0/(:rk + rt)) DESC FETCH FIRST {int(k)} ROWS ONLY"""
    with conn.cursor() as cur:
        cur.execute(sql, q=embed_query(q), kw=_terms(q), rk=rrf_k)
        return [r[0] for r in cur.fetchall()]

**Compute the metrics locally** and lay the three techniques side by side. (This is the same comparison you'd get from a dedicated IR-eval library — we keep it in plain NumPy/pandas so the mechanics are visible.)

In [ ]:
import pandas as pd


def hit_at_k(retrieved, relevant, k):
    return float(any(d in set(relevant) for d in retrieved[:k]))


def mrr(retrieved, relevant):
    for i, d in enumerate(retrieved, 1):
        if d in set(relevant):
            return 1.0 / i
    return 0.0


def score_retriever(fn):
    rows = [(hit_at_k(fn(e["question"]), e["relevant"], 1),
             hit_at_k(fn(e["question"]), e["relevant"], 3),
             mrr(fn(e["question"]), e["relevant"])) for e in RETRIEVAL_EVALSET]
    arr = np.array(rows)
    return {"recall@1": arr[:, 0].mean(), "recall@3": arr[:, 1].mean(), "MRR": arr[:, 2].mean()}


comparison = pd.DataFrame({
    "keyword": score_retriever(retr_keyword),
    "vector": score_retriever(retr_vector),
    "hybrid": score_retriever(retr_hybrid),
}).T.round(3)
print(comparison)

📊 **Visualize Part 1.** The same keyword/vector/hybrid comparison as grouped bars — easier to read at a glance than the table.

In [ ]:
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

ax = comparison.plot.bar(figsize=(7, 4), rot=0, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("Part 1 — Acme retrieval: technique comparison")
ax.set_ylabel("score"); ax.set_ylim(0, 1.08); ax.legend(title="metric", loc="lower right")
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", padding=2, fontsize=8)
plt.tight_layout(); plt.show()

> 💡 **Teachable moment — read the table, not the vibes.** On paraphrased questions, **keyword** recall is low (the words don't overlap), **vector** recall is high (meaning matches), and **hybrid** usually matches or beats vector by also catching exact-term hits. This is a *measurement*, not an opinion — re-run it after any retrieval change and you'll see immediately whether you helped or hurt.

## 1.1 The same evaluation, on the LangSmith platform

The local table is great for a quick look. Putting it on **LangSmith** gives you a versioned, shareable **experiment**: every example, every score, browsable in the UI and comparable across runs. The shape is always the same — **dataset → target → evaluators → `evaluate`**.

In [ ]:
def ensure_dataset(name, examples, description=""):
    """Create the dataset once; reuse it on re-runs (idempotent)."""
    if ls.has_dataset(dataset_name=name):
        return ls.read_dataset(dataset_name=name)
    ds = ls.create_dataset(dataset_name=name, description=description)
    ls.create_examples(dataset_id=ds.id, examples=examples)
    return ds


retrieval_ds = ensure_dataset(
    "acme-retrieval",
    [{"inputs": {"question": e["question"]}, "outputs": {"relevant": e["relevant"]}}
     for e in RETRIEVAL_EVALSET],
    description="Acme Cloud paraphrased questions with the doc_ids that should be retrieved.",
)
print("Dataset ready:", retrieval_ds.name)

The **target** wraps the retriever (here, vector search); the **evaluators** turn our metrics into LangSmith scores using the `(inputs, outputs, reference_outputs)` signature.

In [ ]:
def retrieval_target(inputs: dict) -> dict:
    return {"retrieved": retr_vector(inputs["question"], k=5)}


def recall_at_3(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "recall@3", "score": hit_at_k(outputs["retrieved"], reference_outputs["relevant"], 3)}


def mrr_eval(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "mrr", "score": mrr(outputs["retrieved"], reference_outputs["relevant"])}


retrieval_experiment = ls.evaluate(
    retrieval_target,
    data="acme-retrieval",
    evaluators=[recall_at_3, mrr_eval],
    experiment_prefix="vector-retrieval",
    max_concurrency=4,
)
print("Retrieval experiment complete — open it in the LangSmith UI to browse per-example scores.")

> ### 📌 Key Takeaways — Evaluating Retrieval
> - **Grade the evidence before the answer.** Recall@k and MRR tell you whether the right document is even reachable.
> - A tiny **labeled set** (question → relevant doc_id) is enough to compare retrieval strategies objectively.
> - The LangSmith shape — **dataset → target → evaluators → `evaluate`** — is identical for every kind of eval; only the evaluators change.

# Part 2 — Evaluating RAG Generation

Retrieval can be perfect and the *answer* still wrong — the model might ignore the context, add facts that aren't there, or contradict the docs. Two judgments matter most:

> 💡 **Key Term — the RAG quality pair:**
> - **Correctness** — does the answer match a known-good reference answer?
> - **Groundedness (faithfulness)** — is every claim in the answer supported by the retrieved context, with no hallucinations?
>
> Neither is an exact-string check, so we use an **LLM-as-judge**: Claude reads the answer (and the reference, or the context) and returns a structured verdict.

First, a labeled QA set.

In [ ]:
RAG_EVALSET = [
    {"question": "What is the API rate limit on the Pro plan?",
     "reference": "The Pro plan allows 1,000 requests per minute."},
    {"question": "Which plan do I need for SSO?",
     "reference": "SSO is available on the Enterprise plan."},
    {"question": "What uptime does the Pro SLA guarantee?",
     "reference": "99.9% uptime on Pro (99.99% on Enterprise)."},
    {"question": "How long are backups kept on Enterprise?",
     "reference": "90 days on Enterprise (30 days on Pro)."},
    {"question": "Does the Free plan include 24/7 phone support?",
     "reference": "No. Free uses the community forum; 24/7 support is an Enterprise feature."},
    {"question": "Can I change my data region after creating a project?",
     "reference": "No — the region is fixed at project creation and cannot be changed afterward."},
]
print(len(RAG_EVALSET), "labeled QA examples")

**The Claude judge.** One helper sends a rubric + the material to grade and gets back a structured `{score, reason}` (constrained by `output_config`, so parsing never fails). Each evaluator is a thin wrapper around it.

In [ ]:
JUDGE_SCHEMA = {
    "type": "object",
    "properties": {"score": {"type": "number"}, "reason": {"type": "string"}},
    "required": ["score", "reason"],
    "additionalProperties": False,
}


def claude_judge(rubric: str, material: str) -> dict:
    resp = client.messages.create(
        model=MODEL, max_tokens=400, system=rubric,
        messages=[{"role": "user", "content": material}],
        output_config={"format": {"type": "json_schema", "schema": JUDGE_SCHEMA}},
    )
    return json.loads(text_of(resp))


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    rubric = ("You grade a support answer against a reference. Return score 1 if the answer is "
              "factually consistent with the reference (same key facts; wording may differ), else 0. "
              'Respond as JSON {"score": 0 or 1, "reason": "..."}.')
    material = (f"Question: {inputs['question']}\n\nReference answer: {reference_outputs['reference']}\n\n"
               f"Answer to grade: {outputs['answer']}")
    v = claude_judge(rubric, material)
    return {"key": "correctness", "score": float(v["score"]), "comment": v["reason"]}


def groundedness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    rubric = ("You check for hallucinations. Return score 1 if EVERY factual claim in the answer is "
              "directly supported by the provided context, else 0. "
              'Respond as JSON {"score": 0 or 1, "reason": "..."}.')
    material = f"Context:\n{outputs['context']}\n\nAnswer:\n{outputs['answer']}"
    v = claude_judge(rubric, material)
    return {"key": "groundedness", "score": float(v["score"]), "comment": v["reason"]}


# Sanity-check the judge directly (works without LangSmith) before running the experiment:
print(correctness({"question": "Pro rate limit?"},
                  {"answer": "Pro allows 1,000 requests/minute."},
                  {"reference": "The Pro plan allows 1,000 requests per minute."}))

**The target** runs the real `rag_pipeline` and exposes both the answer and the context (groundedness needs the context). Then we register the dataset and run the experiment.

In [ ]:
def rag_target(inputs: dict) -> dict:
    answer, context = rag_pipeline(inputs["question"])
    return {"answer": answer, "context": context}


ensure_dataset(
    "acme-rag-qa",
    [{"inputs": {"question": e["question"]}, "outputs": {"reference": e["reference"]}} for e in RAG_EVALSET],
    description="Acme Cloud question → reference answer pairs for RAG correctness & groundedness.",
)

rag_experiment = ls.evaluate(
    rag_target,
    data="acme-rag-qa",
    evaluators=[correctness, groundedness],
    experiment_prefix="rag-claude-judge",
    max_concurrency=2,
)
print("RAG experiment complete. Aggregate scores:")
print(rag_experiment.to_pandas()[["feedback.correctness", "feedback.groundedness"]].mean())

📊 **Visualize Part 2.** Mean correctness and groundedness across the QA set (1.0 = every example passed).

In [ ]:
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

rag_scores = rag_experiment.to_pandas()[["feedback.correctness", "feedback.groundedness"]].mean()
rag_scores.index = ["correctness", "groundedness"]
ax = rag_scores.plot.bar(figsize=(5, 4), rot=0, color=["#4C72B0", "#55A868"])
ax.set_title("Part 2 — RAG generation quality (mean)")
ax.set_ylabel("score (1.0 = pass)"); ax.set_ylim(0, 1.08)
ax.bar_label(ax.containers[0], fmt="%.2f", padding=2)
plt.tight_layout(); plt.show()

> 💡 **Teachable moment — correctness vs. groundedness can disagree, and that's the point.** An answer can be *grounded* (faithful to the retrieved context) yet *incorrect* (because retrieval surfaced the wrong context), or *correct* yet *ungrounded* (the model knew the fact but the context didn't contain it — a latent hallucination risk). Tracking both separates a *retrieval* problem from a *generation* problem — so you fix the right thing.

> ### 📌 Key Takeaways — Evaluating RAG Generation
> - Grade generation on **correctness** (vs. a reference) and **groundedness** (supported by context, no hallucination).
> - These are subjective, so use an **LLM-as-judge** — Claude, constrained to a structured `{score, reason}` verdict.
> - LangSmith runs the judge over the whole dataset and aggregates the scores into a comparable **experiment**.

# Part 3 — Evaluating the Workflow Classifier

The Form Factor 3 workflow began with a `classify` step that routes each message. Classification has a **ground truth**, so the evaluators are simple **exact-match** checks — fast, deterministic, and free (no judge LLM needed).

> 💡 **Key Insight — use the cheapest evaluator that fits.** Reach for an LLM-as-judge only when the thing you're scoring is genuinely subjective. When there's a correct label (a category, a number, valid JSON), a plain equality check is faster, cheaper, and not itself a source of error.

In [ ]:
CLASSIFY_EVALSET = [
    {"message": "I was charged twice for my Pro seats this month.",          "category": "billing",         "urgency": "high"},
    {"message": "The API returns a 500 error on the /v1/sync endpoint.",     "category": "technical",       "urgency": "high"},
    {"message": "How do I add a teammate to my workspace?",                  "category": "account",         "urgency": "low"},
    {"message": "It would be great if you supported exporting to Parquet.",  "category": "feature_request", "urgency": "low"},
    {"message": "Just saying thanks, the product is great!",                 "category": "other",           "urgency": "low"},
    {"message": "Webhooks stopped firing and it's blocking our launch.",     "category": "technical",       "urgency": "high"},
]


def classify_target(inputs: dict) -> dict:
    return classify(inputs["message"])


def category_match(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "category_match", "score": float(outputs["category"] == reference_outputs["category"])}


def urgency_match(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "urgency_match", "score": float(outputs["urgency"] == reference_outputs["urgency"])}


ensure_dataset(
    "acme-classification",
    [{"inputs": {"message": e["message"]}, "outputs": {"category": e["category"], "urgency": e["urgency"]}}
     for e in CLASSIFY_EVALSET],
    description="Acme Cloud support messages with gold category + urgency labels.",
)

classify_experiment = ls.evaluate(
    classify_target,
    data="acme-classification",
    evaluators=[category_match, urgency_match],
    experiment_prefix="workflow-classifier",
    max_concurrency=4,
)
print("Classifier experiment complete. Accuracy:")
print(classify_experiment.to_pandas()[["feedback.category_match", "feedback.urgency_match"]].mean())

📊 **Visualize Part 3.** Per-field classifier accuracy — category vs urgency.

In [ ]:
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

clf_scores = classify_experiment.to_pandas()[["feedback.category_match", "feedback.urgency_match"]].mean()
clf_scores.index = ["category", "urgency"]
ax = clf_scores.plot.bar(figsize=(5, 4), rot=0, color=["#C44E52", "#8172B3"])
ax.set_title("Part 3 — Classifier accuracy (exact match)")
ax.set_ylabel("accuracy"); ax.set_ylim(0, 1.08)
ax.bar_label(ax.containers[0], fmt="%.2f", padding=2)
plt.tight_layout(); plt.show()

> ### 📌 Key Takeaways — Evaluating the Workflow Classifier
> - When there's a **ground-truth label**, exact-match evaluators are the right tool — deterministic and cost-free.
> - Per-field scoring (`category` vs. `urgency`) localizes failures: maybe the category is reliable but urgency drifts.
> - The same `evaluate` call powers it — only the dataset and evaluators changed.

# Part 4 — A Rigorous Retrieval Bake-off (BEIR Benchmark)

Part 1 compared retrievers on a *toy* set — 10 paraphrased questions over 12 Acme docs — and the scores **saturated** (vector hit recall@3 = 1.0). That's fine for learning the mechanics, but it can't credibly crown a "best" retriever: with so few documents and questions, the differences are noise.

To settle which retrieval technique wins, we need a real **benchmark**: many documents, many queries, and *expert relevance judgments*. We use **BEIR** — the standard suite for evaluating retrieval — and its **`scifact`** dataset (scientific *claims* as queries, paper abstracts as documents, with human-labeled `qrels` saying which abstracts support each claim).

> 💡 **Key Term — qrels (query relevance judgments):** the ground truth of an IR benchmark — for each query, the set of documents a human marked relevant. Every metric below is computed by comparing a retriever's ranked results against the qrels.

## The Three Classic Retrieval Metrics

We score each retriever with the three metrics every IR practitioner reports. All are computed **@k** (over the top-`k` returned documents).

> 💡 **Precision@k — how *pure* is the result list?**
> $$\text{Precision@}k = \frac{\#\{\text{relevant docs in top-}k\}}{k}$$
> Of the `k` documents you returned, what fraction are actually relevant. High precision = little junk near the top. **Caveat:** when a query has only a few relevant documents (scifact averages ~1), precision is *capped low by construction* — with a single relevant doc, the best possible `Precision@10` is `1/10 = 0.1`. So always read precision relative to how many relevant docs a query *can* have.

> 💡 **Recall@k — did we *find the evidence*?**
> $$\text{Recall@}k = \frac{\#\{\text{relevant docs in top-}k\}}{\#\{\text{all relevant docs}\}}$$
> Of all the documents that *should* have been found, what fraction made it into the top-`k`. This is usually **the most important metric for RAG**: if the one abstract that answers the question isn't retrieved, the generator literally cannot produce a grounded answer — no prompt engineering can recover a document that was never passed in.

> 💡 **NDCG@k — did we *rank them well*?**
> Precision and recall ignore *order* — a relevant doc at rank 1 scores the same as at rank 10. **NDCG (Normalized Discounted Cumulative Gain)** fixes that:
> $$\text{DCG@}k = \sum_{i=1}^{k} \frac{rel_i}{\log_2(i+1)} \qquad \text{NDCG@}k = \frac{\text{DCG@}k}{\text{IDCG@}k}$$
> `rel_i` is the relevance of the doc at rank `i` (1 if relevant, else 0 for binary judgments like scifact). The `log_2(i+1)` denominator **discounts** lower ranks, so a hit at rank 1 is worth more than the same hit at rank 5. Dividing by **IDCG** (the score of the *ideal* ranking — all relevant docs first) normalizes to `[0, 1]`, where **1.0 = a perfect ranking**, making scores comparable across queries.
>
> **Worked example (one relevant doc):** returned at rank 1 → `NDCG@10 = (1/\log_2 2)/(1/\log_2 2) = 1.0`. Returned at rank 3 → `DCG = 1/\log_2 4 = 0.5`, `IDCG = 1.0` → `NDCG@10 = 0.5`. Same *recall* (1.0) in both cases, but very different NDCG — that's the rank-awareness precision and recall miss.

## 4.1 Load the Benchmark

We load `scifact` straight from the Hugging Face Hub with the `datasets` library (no PyTorch needed). One id-hygiene note: corpus/query `_id`s are strings while the qrels store ids as integers, so we join on `str(...)`.

> **Note:** to keep this runnable on a laptop we cap the corpus to `MAX_CORPUS` docs (always including every doc the chosen queries need) and use `N_QUERIES` of the ~300 test queries. Raise them (set `MAX_CORPUS = None`) for the full benchmark on a bigger machine.

In [ ]:
from datasets import load_dataset

N_QUERIES = 60        # scifact has ~300 test queries; cap for a snappy run
MAX_CORPUS = 2000     # scifact has 5,183 docs; cap for memory/runtime (set None for the full corpus)

corpus = load_dataset("BeIR/scifact", "corpus", split="corpus")
queries = load_dataset("BeIR/scifact", "queries", split="queries")
qrels = load_dataset("BeIR/scifact-qrels", split="test")

qtext = {str(q["_id"]): (q["text"] or "") for q in queries}
qrels_by_q = {}
for r in qrels:                                    # {query_id: {doc_id: relevance}}
    qrels_by_q.setdefault(str(r["query-id"]), {})[str(r["corpus-id"])] = int(r["score"])

corpus_by_id = {str(d["_id"]): d for d in corpus}
test_qids = [qid for qid in qrels_by_q if qid in qtext][:N_QUERIES]

# Corpus = every doc relevant to the chosen queries (so recall is achievable) + distractors up to the cap.
relevant = {d for qid in test_qids for d in qrels_by_q[qid] if d in corpus_by_id}
distractors = [cid for cid in corpus_by_id if cid not in relevant]
keep = list(relevant) + distractors
if MAX_CORPUS:
    keep = keep[:max(MAX_CORPUS, len(relevant))]
beir_docs_data = [(cid, corpus_by_id[cid]["title"], corpus_by_id[cid]["text"] or "") for cid in keep]
print(f"{len(test_qids)} test queries | {len(beir_docs_data)} corpus docs "
      f"({len(relevant)} relevant + {len(beir_docs_data) - len(relevant)} distractors)")

## 4.2 Embed and Ingest the Corpus

We reuse the same `fastembed` model and Oracle table pattern as the rest of the notebook.

> 💡 **Teachable moment — `batch_size` matters on constrained machines.** scifact's abstracts are long; embedding them in large batches (fastembed's default) can spike memory and get the process OOM-killed, especially alongside the Oracle container. We pass a small `batch_size=8` (and let `fastembed` run single-process) to keep peak memory low. This is the kind of operational detail that decides whether an eval *runs at all*.

In [ ]:
beir_vectors = np.array(
    [_unit(v) for v in embedder.embed([f"{t}. {x[:1000]}" for _, t, x in beir_docs_data], batch_size=8)],
    dtype=np.float32,
)
bdim = int(beir_vectors.shape[1])

beir_ddl = f"""
BEGIN EXECUTE IMMEDIATE 'DROP TABLE beir_docs CASCADE CONSTRAINTS PURGE';
EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF; END;
/
CREATE TABLE beir_docs (doc_id VARCHAR2(64) PRIMARY KEY, title VARCHAR2(1000),
    content VARCHAR2(4000), embedding VECTOR({bdim}, FLOAT32))
"""
with conn.cursor() as cur:
    for stmt in beir_ddl.split("/"):
        if stmt.strip():
            cur.execute(stmt)
    try:
        cur.execute("DROP INDEX beir_text_idx")
    except oracledb.DatabaseError:
        pass
    cur.execute("CREATE INDEX beir_text_idx ON beir_docs(content) "
                "INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')")
    cur.executemany(
        "INSERT INTO beir_docs (doc_id, title, content, embedding) VALUES (:1, :2, :3, :4)",
        [(cid, t.encode("utf-8")[:990].decode("utf-8", "ignore"), x.encode("utf-8")[:3900].decode("utf-8", "ignore"), array.array("f", v))
         for (cid, t, x), v in zip(beir_docs_data, beir_vectors.astype(np.float32).tolist())],
    )
conn.commit()
print(f"Ingested {len(beir_docs_data)} BEIR docs (VECTOR dim={bdim}).")

## 4.3 The Metrics and the Retrievers

The three metric functions implement the formulas above (binary relevance, so `rel_i ∈ {0, 1}`). The three retrievers are the same keyword / vector / hybrid techniques, now over the `beir_docs` table; they return ranked `doc_id`s. (`_terms` and `embed_query` are reused from Part 1.)

In [ ]:
import math


def precision_at_k(retrieved, relevant, k=10):
    return sum(1 for d in retrieved[:k] if relevant.get(d, 0) > 0) / k


def recall_at_k(retrieved, relevant, k=10):
    rel = {d for d, s in relevant.items() if s > 0}
    return len(rel & set(retrieved[:k])) / len(rel) if rel else 0.0


def ndcg_at_k(retrieved, relevant, k=10):
    dcg = sum(1.0 / math.log2(i + 2) for i, d in enumerate(retrieved[:k]) if relevant.get(d, 0) > 0)
    n_rel = sum(1 for s in relevant.values() if s > 0)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(k, n_rel)))   # ideal: relevant docs first
    return dcg / idcg if idcg > 0 else 0.0


def beir_keyword(question, k=10):
    sql = f"""SELECT doc_id FROM beir_docs WHERE CONTAINS(content, :kw, 1) > 0
              ORDER BY SCORE(1) DESC FETCH FIRST {int(k)} ROWS ONLY"""
    with conn.cursor() as cur:
        cur.execute(sql, kw=_terms(question))
        return [r[0] for r in cur.fetchall()]


def beir_vector(question, k=10):
    sql = f"""SELECT doc_id FROM beir_docs
              ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE) FETCH FIRST {int(k)} ROWS ONLY"""
    with conn.cursor() as cur:
        cur.execute(sql, q=embed_query(question))
        return [r[0] for r in cur.fetchall()]


def beir_hybrid(question, k=10, per_list=50, rrf_k=60):
    sql = f"""
        WITH vec AS (SELECT doc_id, ROW_NUMBER() OVER (ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE)) r
                     FROM beir_docs ORDER BY VECTOR_DISTANCE(embedding, :q, COSINE) FETCH FIRST {int(per_list)} ROWS ONLY),
             txt AS (SELECT doc_id, ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) r
                     FROM beir_docs WHERE CONTAINS(content, :kw, 1) > 0
                     ORDER BY SCORE(1) DESC FETCH FIRST {int(per_list)} ROWS ONLY),
             fused AS (SELECT COALESCE(v.doc_id, t.doc_id) doc_id, NVL(v.r, 999999) rv, NVL(t.r, 999999) rt
                       FROM vec v FULL OUTER JOIN txt t ON t.doc_id = v.doc_id)
        SELECT doc_id FROM fused ORDER BY (1.0/(:rk + rv) + 1.0/(:rk + rt)) DESC FETCH FIRST {int(k)} ROWS ONLY"""
    with conn.cursor() as cur:
        cur.execute(sql, q=embed_query(question), kw=_terms(question), rk=rrf_k)
        return [r[0] for r in cur.fetchall()]

## 4.4 The Bake-off

Retrieve once per (technique, query) and average precision@10, recall@10, and NDCG@10 over the benchmark.

In [ ]:
import pandas as pd

K = 10
rows = {}
for name, retr in [("keyword", beir_keyword), ("vector", beir_vector), ("hybrid", beir_hybrid)]:
    P = R = N = 0.0
    for qid in test_qids:
        got, rel = retr(qtext[qid], k=K), qrels_by_q[qid]
        P += precision_at_k(got, rel, K)
        R += recall_at_k(got, rel, K)
        N += ndcg_at_k(got, rel, K)
    n = len(test_qids)
    rows[name] = {f"precision@{K}": P / n, f"recall@{K}": R / n, f"NDCG@{K}": N / n}

beir_table = pd.DataFrame(rows).T.round(3)
print(beir_table)

📊 **Visualize Part 4.** The bake-off at a glance — watch vector and hybrid pull ahead of keyword on recall@10 and NDCG@10 (precision@10 sits near 0.1 because scifact queries have ~1 relevant doc).

In [ ]:
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

ax = beir_table.plot.bar(figsize=(8, 4), rot=0, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("Part 4 — BEIR scifact bake-off: precision / recall / NDCG @10")
ax.set_ylabel("score"); ax.set_ylim(0, 1.0); ax.legend(title="metric", loc="upper right")
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", padding=2, fontsize=8)
plt.tight_layout(); plt.show()

> 💡 **Teachable moment — now the winner is real.** Unlike the toy set in Part 1, these scores are **not saturated** and the gaps are meaningful: vector search beats keyword on NDCG (semantics matter for paraphrased scientific claims), and hybrid typically edges out pure vector by also catching exact-term matches. Note **precision@10 looks low (~0.1)** — that's expected, not a bug: scifact queries average about one relevant doc, so `precision@10` is capped near `0.1`. Here **recall@10 and NDCG@10 are the metrics that discriminate** — exactly why you report all three and read each in context.

## 4.5 Head-to-Head on LangSmith

Finally, run each retriever as its own LangSmith **experiment** over one shared dataset. Because they share the dataset, LangSmith's compare view lines them up side by side — the definitive, shareable "which retriever wins" artifact.

In [ ]:
beir_ds = ensure_dataset(
    "beir-scifact-retrieval",
    [{"inputs": {"question": qtext[qid]}, "outputs": {"relevant": qrels_by_q[qid]}} for qid in test_qids],
    description="BEIR scifact: scientific-claim queries with expert relevance judgments (qrels).",
)


def beir_evaluators(k=10):
    def precision(inputs, outputs, reference_outputs):
        return {"key": f"precision@{k}", "score": precision_at_k(outputs["retrieved"], reference_outputs["relevant"], k)}

    def recall(inputs, outputs, reference_outputs):
        return {"key": f"recall@{k}", "score": recall_at_k(outputs["retrieved"], reference_outputs["relevant"], k)}

    def ndcg(inputs, outputs, reference_outputs):
        return {"key": f"ndcg@{k}", "score": ndcg_at_k(outputs["retrieved"], reference_outputs["relevant"], k)}

    return [precision, recall, ndcg]


for name, retr in [("keyword", beir_keyword), ("vector", beir_vector), ("hybrid", beir_hybrid)]:
    ls.evaluate(
        lambda inputs, _r=retr: {"retrieved": _r(inputs["question"], k=K)},
        data="beir-scifact-retrieval",
        evaluators=beir_evaluators(K),
        experiment_prefix=f"beir-{name}",
        max_concurrency=4,
    )
    print(f"  experiment 'beir-{name}-*' uploaded")
print("Three head-to-head experiments are in LangSmith — open the dataset and compare them.")

> ### 📌 Key Takeaways — The Retrieval Bake-off
> - **Toy datasets saturate.** To pick a "best" retriever you need a real benchmark with many docs, many queries, and **qrels** (expert relevance labels). BEIR is the standard.
> - **Report all three metrics:** **precision@k** (purity), **recall@k** (coverage — the key RAG metric), and **NDCG@k** (rank-aware quality). Read each in context — e.g. precision is capped low when queries have few relevant docs.
> - Running each technique as a **LangSmith experiment on a shared dataset** turns "which retriever is best?" into a side-by-side comparison you can version and re-run after any change.
> - Operational reality matters: a small `batch_size` is what lets the embedding step finish on a constrained machine.

# Where to Next?

You now have a measured version of the ladder: **retrieval** scored with IR metrics, **RAG generation** judged by Claude for correctness and groundedness, and the **workflow classifier** checked against gold labels — all as LangSmith experiments.

> 💡 **Key Insight — evaluation is a habit, not a milestone.** The value compounds when you run these experiments *every time you change something*: a new model, a tweaked prompt, a different `k` or retrieval strategy. LangSmith keeps the history, so "did this change help?" becomes a diff between two experiments instead of a guess.

**Take it further:**

- **Regression-test in CI** — fail a build if `correctness` or `recall@3` drops below a threshold.
- **Grow the datasets** — add every real failure you find as a new example, so it's caught next time.
- **Online evaluation** — attach evaluators to *production traces* (via the wrapped Anthropic client) to monitor live quality, not just the offline set.
- **Pairwise / A-B experiments** — compare two prompts or models head-to-head on the same dataset.

**References:**

- **[LangSmith Evaluation docs](https://docs.langchain.com/langsmith/evaluation)** — datasets, evaluators, experiments
- **[LangSmith `evaluate` reference](https://reference.langchain.com/python/langsmith/evaluation)** — the Python SDK surface
- **[Claude API — structured outputs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs)** — the JSON-schema technique behind the judge
- **[Oracle AI Vector Search](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)** — the retrieval engine under test